# Amypred-FRL analysis — Aβ42 Variants

In [2]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BG       = '#0a0c14'
SURFACE  = '#111520'
MUTED    = '#64748b'
TEXT     = '#e2e8f0'
WT_COLOR = '#4ff7c0'
C_DESTAB = '#3b82f6'
C_NEUTRAL= '#64748b'
C_STAB   = '#ef4444'
THRESH   = 0.003

# ── Loading ──────────────────────────────────────────────────
df = pd.read_csv('../predictions/amypred/amypred-frl.csv', sep=';', encoding='utf-8-sig')
df['short'] = df['Name'].str.replace(r'_Abeta_?42$', '', regex=True)
wt_p = df[df['Name'] == 'Wildtype_Abeta_42']['Probability'].iloc[0]
df['ΔP'] = df['Probability'] - wt_p

def pc(d):
    if d < -THRESH: return C_DESTAB
    if d >  THRESH: return C_STAB
    return C_NEUTRAL

n = len(df)
fs = max(4.2, min(6.5, 8 * 20 / n))

In [ ]:
# PLOT 1 — Lollipop: ΔProbability vs Wildtype

df_lol = df.sort_values('ΔP', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 8), facecolor=BG)
ax.set_facecolor(SURFACE)

for i in range(n):
    if i % 2 == 0:
        ax.axhspan(i - 0.5, i + 0.5, color='white', alpha=0.018, zorder=0)

ax.axvline(0, color=WT_COLOR, lw=1.0, linestyle='--', alpha=0.55)
ax.text(0, n + 0.1, f'WT\n{wt_p:.3f}', ha='center', va='bottom',
        color=WT_COLOR, fontsize=6.5, fontfamily='monospace', fontweight='bold')

ax.axvspan(-0.045, -THRESH, color=C_DESTAB, alpha=0.055)
ax.axvspan(THRESH,  0.028,  color=C_STAB,   alpha=0.04)

for i, (_, row) in enumerate(df_lol.iterrows()):
    c = pc(row['ΔP'])
    ax.plot([0, row['ΔP']], [i, i], color=c, lw=0.8, alpha=0.45)
    ax.scatter(row['ΔP'], i, color=c, s=22, zorder=3, linewidths=0)
    xoff = row['ΔP'] + (0.001 if row['ΔP'] >= 0 else -0.001)
    ax.text(xoff, i, f"{row['ΔP']:+.3f}",
            va='center', ha='left' if row['ΔP'] >= 0 else 'right',
            fontsize=4.8, fontfamily='monospace', color=c, alpha=0.75)

ax.set_yticks(range(n))
ax.set_yticklabels(df_lol['short'].values, fontsize=fs, fontfamily='monospace')
for tick, (_, row) in zip(ax.get_yticklabels(), df_lol.iterrows()):
    tick.set_color(pc(row['ΔP'])); tick.set_alpha(0.9)

ax.tick_params(axis='y', length=0, pad=3)
ax.tick_params(axis='x', colors=MUTED, labelsize=7)
for sp in ax.spines.values(): sp.set_visible(False)
ax.set_xlim(-0.052, 0.038)
ax.set_ylim(-1, n)

n_d = (df_lol['ΔP'] < -THRESH).sum()
n_n = (df_lol['ΔP'].abs() <= THRESH).sum()
n_s = (df_lol['ΔP'] > THRESH).sum()
lp = [
    mpatches.Patch(facecolor=C_DESTAB,  label=f'Decrease amyloidogenicity (Δ<0)  · n={n_d}'),
    mpatches.Patch(facecolor=C_NEUTRAL, label=f'Neutral  · n={n_n}'),
    mpatches.Patch(facecolor=C_STAB,    label=f'Increase amyloidogenicity (Δ>0)  · n={n_s}'),
]
ax.legend(handles=lp, loc='lower right', frameon=True, framealpha=0.15,
          edgecolor=MUTED, facecolor=SURFACE, fontsize=7,
          labelcolor=TEXT, handlelength=0.9)

ax.set_xlabel(
    'Δ Amyloid Probability  (mutant − WT)\n'
    '< 0  →  lower probability of amyloid formation',
    color=MUTED, fontsize=8, fontfamily='monospace', labelpad=8)
ax.set_title(
    f'AmyPred-FRL  ·  Δ Probability vs Wildtype\nWT = {wt_p:.3f}',
    color='white', fontsize=11, fontfamily='monospace', fontweight='bold', pad=10)

plt.tight_layout()
plt.show()
plt.savefig('amypred_lollipop.png', dpi=200, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
print('Saved: amypred_lollipop.png')
plt.close()

In [ ]:
# PLOT 2 — Bar chart: absolute probabilities

df_bar = df.sort_values('Probability', ascending=True).reset_index(drop=True)
left   = df_bar['Probability'].min() - 0.005

fig, ax = plt.subplots(figsize=(10, 8), facecolor=BG)
ax.set_facecolor(SURFACE)

for i, (_, row) in enumerate(df_bar.iterrows()):
    is_wt = row['Name'] == 'Wildtype_Abeta_42'
    c = WT_COLOR if is_wt else pc(row['ΔP'])
    if i % 2 == 0:
        ax.axhspan(i - 0.5, i + 0.5, color='white', alpha=0.018, zorder=0)
    ax.barh(i, row['Probability'] - left, left=left,
            color=c, alpha=1.0 if is_wt else 0.78, height=0.6, zorder=2)
    ax.text(row['Probability'] + 0.0005, i, f"{row['Probability']:.3f}",
            va='center', ha='left', color=c,
            fontsize=4.8, fontfamily='monospace', alpha=0.85)

ax.axvline(wt_p, color=WT_COLOR, lw=1.0, linestyle='--', alpha=0.55)
ax.text(wt_p, n + 0.1, f'WT\n{wt_p:.3f}', ha='center', va='bottom',
        color=WT_COLOR, fontsize=6.5, fontfamily='monospace', fontweight='bold')

ax.set_yticks(range(len(df_bar)))
ax.set_yticklabels(df_bar['short'].values, fontsize=fs, fontfamily='monospace')
for tick, (_, row) in zip(ax.get_yticklabels(), df_bar.iterrows()):
    is_wt = row['Name'] == 'Wildtype_Abeta_42'
    tick.set_color(WT_COLOR if is_wt else pc(row['ΔP']))
    tick.set_alpha(0.9)

ax.tick_params(axis='y', length=0, pad=3)
ax.tick_params(axis='x', colors=MUTED, labelsize=7)
for sp in ax.spines.values(): sp.set_visible(False)
ax.set_xlim(left, 0.985)
ax.set_ylim(-1, n)

ax.set_xlabel('Amyloid Probability',
              color=MUTED, fontsize=8, fontfamily='monospace', labelpad=8)
ax.set_title(
    'AmyPred-FRL  ·  Absolute probabilities\n'
    'Sorted from least to most amyloidogenic',
    color='white', fontsize=11, fontfamily='monospace', fontweight='bold', pad=10)

plt.tight_layout()
plt.show()
plt.savefig('amypred_bars.png', dpi=200, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
print('Saved: amypred_bars.png')
plt.close()